# SDV (Syntetic data Vault)
---
[[site]](https://docs.sdv.dev/sdv)

SDV - библиотека с реализациями методов для работы с генерацией синтетических данных. Развивается как проект MIT с 2016 года

### Обзор
Библиотека предоставляет доступ к небольшому набору моделей-генераторов, которые обучаются (метод fit) и генерируют (метод sample). 

Из важных возможностей:
- работа с мультитабличными датасетами
- работа с PII - делает новые данные
- возможность добавлять свои ограничения и бизнес-правила
- есть несколько встроеных механик проверки качества генерации

### Данные

Работает с любыми датасетами формата pandas DataFrame. Там и предполагается загрузка.

Ещё можно сразу скопом загрузить (если датасет, наример, хранится разбитым на куски)
```
datasets = load_csvs(folder_name='my_folder/', read_csv_parameters={...})
```

Также есть набор преднастроенных датасетов для иллюстрации работы библиотеки
```
import sdv.datasets.demo
```

Можно загрузить интересующий датасет по имени. На выходе таблица + её метаданные
```
real_data, metadata = download_demo(dataset_name='fake_hotels')
```

Датасеты организованы по типу: 
- мультитабличные (55 штук)
- монотабличные (22 штук)
- временные ряды (32 штук)

Списки доступных датасетов можно посмотреть по get_available_demos()
```
get_available_demos(modality='multi_table')
```



### Метаданные
Данные = то, что содержится в строках таблицы. Метаданные = информация о типах данных, которые датасет содержит

В SDV для перед генерацией новых данных необходимо каждому датасету сопоставить свои метаданные

Класс Metadata - это просто словарь (обернутый неколькими методами), описывающий структуру датасета

Ниже о том, как его создать, обновить, вывести

Создать метаданные можно одним из нескольких способов:
- сделать автоматический inference по готовой таблице<br>```Metadata.detect_from_dataframe()```<br>способ неидеальный. при определении типов и ключей бывают ошибки, поэтому их можно отключить - будет установлены в unknown<br><br>
- собрать вручную<br>```metyadata = SingleTableMetadata()```<br><br>
- загрузить уже преднастроенные (как делается в случае с download_demo и готовыми примерами)<br>```load_from_json()```

Можно посмотреть, что внутри 
- напечатать через print
- нарисовать блок-схему (формата logical data model) методом __visualize()__
- достать в виде словаря __metadata.to_dict()__

Какие манипуляции можно делать с метаданными
- add_table / remove_table - добавить в метаданные запись о таблице
- add_column / remove_columns - добавить / удалить в метаданные данные о поле таблицы
- add_column_relationship - объединить поля таблицы в группу, если они семантически описывают один и тот же концепт
- set_primary_key / remove_proimaty_kjey - установить первичный ключ
- add_alternative_keys - добавить другие уникальные ключи
- add_relationship / remove_realtionship - добавить / уддалить связь между таблицами
- update_column - обновить метаданные для выбранного поля
- update_columns - то же, но сразу для набора полей
- update_column_metadata - то же но не кокнретные поля, а сразу в JSON-подобном формате
- set-sequence_ky
- set_sequence_index
```
metadata.update_column(
    column_name='start_date',
    sdtype='datetime',
    datetime_format='%Y-%m-%d')
```

Поскольку SDV предполагает и ручную работу с метаданными, перед использованием полезно делать валидацию
- validate - проверяет на непротиворечивость
- validat_data - проверяет метаданные на соответствие данным
- validate_table - то же, но для конкретной таблицы (если их несколько)

Сохранить / загрпузить можно через ```save()``` и ```load()```

Пример структуры объекта с метадаными:
- tables - открываем перечисление таблиц датасета
    - "table_A" - имя описываемой таблицы
        - columns - открываем перечисление полей таблицы
            - name - нзвание поля
            - sdtype - тип поля (тут не только базовые типы, но и специальные)
                - params - доп параметры
                    - регулярное выражение, описывающее возможные значения
                    - является ли поле PII
                    - прочее
        - primary keys - какие поля из перечисленных однозначно идентифицируют запись
- relations - связи между таблицами

### Типы полей
В SDV называются sdtypes, это более выскоуровнеовое понятие типа

Примеры:
- numeric
- categorical
- boolean
- address
- email
- credit_card
- id

Категориальные признаки кодируются в непрерывные через Frequency Encoding

<img src='img/tv2.png' width=300>

## Генерация

Здесь всё просто - создаем экземпляр нужного нам синтезатора, обучаем его на данных через ```fit()``` и генерируем новые данные через ```sample(scale=2)```

Параметр scale задает, сколько генерировать

## Доступные генераторы
- для одной таблицы
  - параметричееские
    - __GaussianCopulaSynthesizer__
  - нейросетевые
    - CTGAN
    - TVAE
    - CopulaGAN
- для нескольких таблиц
  - HMASynthesize
- для Time Series данных
  - PARSynthesier

Есть некоторые из платной версии:
- DayZSynthesier - генерация только на метаданных

### Выбор локали 
PII данные заменяются на фейки. В конструкторе генератора можно указать нужные локали
```
locales = ['Ru-ru']
```

### Customization
Можно выбирать распределение

### Conditional generation
Можно





### Evaluation

После генерации нужно проверить что сгенерированным можно пользоваться

from sdv.evaluation.multi_table import run_diagnostic

Некоторые доступные методы:
- run_diagnistic() - проверяет выполнение ограничений (уникальность ключей,  и тп)
- evaluate_quality() - сравнивает близость fake vs real, выводит score [0,100]
- get_column_plot() - выводит график сравнения плотностей fake vs real для выбранного поля
- get_column_pair_plot() - то же, но для набора полей (box-whiskers диаграммы)
- QualityReport

Полезно вывести отдельные компоненты проверки
- ColumnShapes - сравнивает marginal распределения
- ParentChild
- Cardinality
- Column Pair trends

При сравнении используется две метрики
- KS (Kolmogorov-Smirnov)<br>
- TV (Total Variance)<br>

Метрика Колмогорова-Смирнова используется для проверки, что два сэмпла данных принадлежат одному распрделеения. Она сравнивает CDF двух величин и возвращает максмимальное расхождение между ними:

<img src='img/ks.png' width=300>

Total Variation - еще одна простая метрика сравнения двух распределений. Считается как половина площади между двумя распредедлениями

<img src='img/tv.png' width=300>

В SDV используется только для сравнения категориальных фичей

### Копулы

Когда поля датасета независимы, их можно генерировать по-отдельности. В реальности большинство полей скоррелированы, это нужно как-то учитывать

Идея копул в том, что моделирование совместного распределение двух переменных декомпозируется на два независимых шага:
1. подбор marginal распределений каждого признака
2. подбор копулы

В каком-то смысле копула - это обобщение понятия корреляция. Корреляция задает только коэффициент линейной зависимости (интегрально), а копула задает соотношение в каждой точке (u,v)

In [8]:
from sdv.datasets.demo import download_demo

real_data, metadata = download_demo(
    modality='multi_table',
    dataset_name='fake_hotels'
)

ModuleNotFoundError: No module named 'sdv'

Минимальный пример

In [9]:
from sdv.single_table import GaussianCopulaSynthesizer

# Выбираем подходящий генератор и инициализируем его метаданными нашего датасета
synthesizer = GaussianCopulaSynthesizer(metadata)

# Подаем на вход данные и обучаем на них 
synthesizer.fit(real_data)

# Далее можем сэмплировать сколько нам нужно
synthetic_data = synthesizer.sample(num_rows=100)

ModuleNotFoundError: No module named 'sdv'